# Training Algorithms — Optimizer Comparison (Polished)

Compares SGD, SGD+Momentum, and RMSProp on a simple FashionMNIST classifier.

**Key concepts:** SGD, Momentum, RMSProp, training loop, accuracy evaluation.

**Notes:**
- Fresh model is created for each optimizer so results are comparable.
- Uses CrossEntropyLoss (preferred over Softmax + NLLLoss).
- Subsets keep runtime reasonable.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## Dataset

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

full_train = datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
full_test = datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)

train_subset, _ = random_split(full_train, [8000, len(full_train)-8000])
test_subset, _ = random_split(full_test, [2000, len(full_test)-2000])

train_loader = DataLoader(train_subset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_subset, batch_size=32, shuffle=False)

## Model & Training Helper

In [ ]:
class FashionNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 10)
        )
    def forward(self, x):
        return self.network(x)

def train_and_eval(optimizer_class, optimizer_kwargs, epochs=10, name="Optimizer"):
    """Train a fresh model and return loss history + test accuracy."""
    model = FashionNN().to(device)
    optimizer = optimizer_class(model.parameters(), **optimizer_kwargs)
    loss_fn = nn.CrossEntropyLoss()
    losses = []
    for epoch in range(epochs):
        model.train()
        running = 0.0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = loss_fn(model(images), labels)
            loss.backward()
            optimizer.step()
            running += loss.item()
        losses.append(running / len(train_loader))
        print(f"{name} Epoch {epoch+1}/{epochs}, Loss: {losses[-1]:.4f}")

    model.eval()
    correct = total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            _, pred = torch.max(model(images), 1)
            total += labels.size(0)
            correct += (pred == labels).sum().item()
    acc = 100 * correct / total
    print(f"{name} Accuracy: {acc:.2f}%\n")
    return losses, acc

sgd_losses, sgd_acc = train_and_eval(optim.SGD, {'lr': 0.01}, name="SGD")
mom_losses, mom_acc = train_and_eval(optim.SGD, {'lr': 0.01, 'momentum': 0.9}, name="SGD+Momentum")
rms_losses, rms_acc = train_and_eval(optim.RMSprop, {'lr': 0.001, 'alpha': 0.9}, name="RMSProp")

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(sgd_losses, label=f"SGD ({sgd_acc:.1f}%)")
plt.plot(mom_losses, label=f"SGD+Mom ({mom_acc:.1f}%)")
plt.plot(rms_losses, label=f"RMSProp ({rms_acc:.1f}%)")
plt.xlabel("Epoch")
plt.ylabel("Training Loss")
plt.legend()
plt.title("Optimizer Comparison on FashionMNIST")
plt.grid(True, alpha=0.3)
plt.show()